# 🧹 User Data Cleaning & Silver Promotion

> **Layer Transition:** Bronze Final → Silver Cleaned  
> **Source Table:** `yelp_poc_uc.bronze_final.yelp_user`  
> **Target Table:** `yelp_poc_uc.silver_cleaned.yelp_user`  
> **DQ Reference:** _(pending — Deep EDA not yet complete)_

This notebook implements the **Bronze → Silver Cleaned data population** for the Yelp User dataset.

---

> ## 🚨 IMPORTANT — THIS NOTEBOOK IS NOT COMPLETE
>
> **Deep EDA has not been performed on the User dataset yet.** Without a thorough data-quality assessment, the full set of cleaning transformations, DQ issue resolutions, and schema decisions cannot be finalized.
>
> **Current scope (temporary):** This notebook exists solely to **propagate `user_id` values from Bronze Final into the Silver Cleaned `yelp_user` table** so that downstream tables with a foreign-key dependency on `user_id` — specifically **`yelp_review`** and **`yelp_tip`** — can have their deferred FK constraints enabled.
>
> Once the User Deep EDA is complete, this notebook will be expanded with the full cleaning logic, column-level transformations, and DQ issue resolutions — following the same pattern as `02_Checkin_Cleaning`.

---

> **⚠️ Prerequisites — Run DDL Queries First:**  
> Before running this notebook, execute the DDL query **`05_User_Silver_Cleaned`** to create the empty `yelp_user` table with PK constraints in `yelp_poc_uc.silver_cleaned`.  
> This notebook then **populates** the table created by the DDL. The DDL is idempotent and can be re-run safely.

## 📍 Pipeline Context

| Step | Notebook | UC Location | Layer |
| ---: | --- | --- | --- |
| 1 | `03_Extract_Yelp_Dataset_from_Kaggle` | `/Volumes/yelp_poc_uc/bronze_landing/landing_zone/yelp_textual_dataset/` | Landing Zone |
| 2 | `01_Landing_to_Staging_Schema_Validator` | `yelp_poc_uc.bronze_staging` | Bronze Staging |
| 3 | `02_Staging_Quality_Gate_and_Promotion` | `yelp_poc_uc.bronze_final.yelp_user` | Bronze Final |
| 4 | `5.1_User_Basic_Profile` | _(reads bronze_final)_ | Basic EDA |
| 5 | _User Deep EDA_ | _(not yet created)_ | Deep EDA / DQ Assessment |
| 5.1 | `05_User_Silver_Cleaned` _(DDL query)_ | `yelp_poc_uc.silver_cleaned.yelp_user` | Schema & Table Creation |
| **6** | **`05_User_Silver_Cleaning` ◄ THIS NOTEBOOK** | **`yelp_poc_uc.silver_cleaned.yelp_user`** | **Silver** |
| 7 | Gold Aggregation _(planned)_ | `yelp_poc_uc.gold` _(planned)_ | Gold |

In [0]:
# ============================================================
# 🔧 Environment Setup — Prerequisite Check
# ============================================================
# PREREQUISITES (run this DDL query BEFORE this notebook):
#   1. 05_User_Silver_Cleaned — creates the yelp_user table
#      in the silver_cleaned schema with PK constraint
#
# This notebook POPULATES the table only.
# No schema creation is needed here.

# Verify the target table exists (created by DDL query)
try:
    tbl_info = spark.sql("DESCRIBE TABLE yelp_poc_uc.silver_cleaned.yelp_user").collect()
    col_count = len([r for r in tbl_info if r['col_name'] and not r['col_name'].startswith('#')])
    print("✅ Target table exists: yelp_poc_uc.silver_cleaned.yelp_user")
    print(f"   Columns: {col_count}")
except Exception as e:
    raise RuntimeError(
        "❌ Target table yelp_poc_uc.silver_cleaned.yelp_user does not exist!\n"
        "   Run the DDL query first:\n"
        "   1. 05_User_Silver_Cleaned\n"
        f"   Error: {e}"
    )

---

## 🔧 Temporary Population — user_id Only

> **🚨 Only the `user_id` primary key is populated.** All other columns remain NULL. This is the minimum needed to satisfy the deferred FK dependencies on `user_id` from `yelp_review` and `yelp_tip`.

Once the User Deep EDA is complete, this section will be replaced with the full cleaning pipeline that populates all columns with type corrections (e.g. `yelping_since` → TIMESTAMP, `elite` → ARRAY\<INT\>, `friends` → ARRAY\<STRING\>).

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import ArrayType, IntegerType, StringType

# ============================================================
# Populate Silver Cleaned — user_id Only
# ============================================================
# Temporary: only the PK column is populated.
# All other columns are explicitly set to NULL.
# This satisfies the FK dependency for yelp_review & yelp_tip.
# Full column population will follow after Deep EDA.

df_bronze = spark.read.table("yelp_poc_uc.bronze_final.yelp_user")

bronze_count = df_bronze.count()
print(f"📥 Bronze source: {bronze_count:>12,} rows")

# Build DataFrame matching the full silver DDL schema:
# user_id populated, all other columns NULL
df_silver = (
    df_bronze
    .select(
        col("user_id"),
        lit(None).cast("string").alias("name"),
        lit(None).cast("bigint").alias("review_count"),
        lit(None).cast("timestamp").alias("yelping_since"),
        lit(None).cast("bigint").alias("useful"),
        lit(None).cast("bigint").alias("funny"),
        lit(None).cast("bigint").alias("cool"),
        lit(None).cast(ArrayType(IntegerType())).alias("elite_years"),
        lit(None).cast(ArrayType(StringType())).alias("friends"),
        lit(None).cast("bigint").alias("fans"),
        lit(None).cast("double").alias("average_stars"),
        lit(None).cast("bigint").alias("compliment_hot"),
        lit(None).cast("bigint").alias("compliment_more"),
        lit(None).cast("bigint").alias("compliment_profile"),
        lit(None).cast("bigint").alias("compliment_cute"),
        lit(None).cast("bigint").alias("compliment_list"),
        lit(None).cast("bigint").alias("compliment_note"),
        lit(None).cast("bigint").alias("compliment_plain"),
        lit(None).cast("bigint").alias("compliment_cool"),
        lit(None).cast("bigint").alias("compliment_funny"),
        lit(None).cast("bigint").alias("compliment_writer"),
        lit(None).cast("bigint").alias("compliment_photos"),
    )
)

# Overwrite to make this notebook idempotent (re-runnable)
(
    df_silver
    .write
    .mode("overwrite")
    .insertInto("yelp_poc_uc.silver_cleaned.yelp_user")
)
print(f"✅ Silver Cleaned table populated (user_id only): yelp_poc_uc.silver_cleaned.yelp_user")

In [0]:
# ============================================================
# ✅ Post-Write Validation
# ============================================================
# Read back from the persisted table to verify correctness.

silver_table = spark.read.table("yelp_poc_uc.silver_cleaned.yelp_user")
sv_count = silver_table.count()

# 1. Row count reconciliation
print("🔍 Validation Results:")
print(f"   Bronze rows:  {bronze_count:>12,}")
print(f"   Silver rows:  {sv_count:>12,}")
assert sv_count == bronze_count, f"❌ Row count mismatch! Bronze={bronze_count}, Silver={sv_count}"
print("   ✅ Row counts match")

# 2. PK uniqueness check
distinct_ids = silver_table.select("user_id").distinct().count()
assert distinct_ids == sv_count, f"❌ Duplicate user_ids! Distinct={distinct_ids}, Total={sv_count}"
print(f"   ✅ user_id is unique ({distinct_ids:,} distinct = {sv_count:,} total)")

# 3. Null check on PK
null_ids = silver_table.filter(col("user_id").isNull()).count()
assert null_ids == 0, f"❌ {null_ids} null user_ids found!"
print(f"   ✅ No null user_ids")

print(f"\n🎉 Validation passed — {sv_count:,} users ready in silver_cleaned")